# Machine Learning — Appointment No-Show Prediction

**Project:** Healthcare Analytics & Predictive Modeling  
**Author:** Radha Yadav  

---

## Objective
Predict which patients are likely to miss their appointments using supervised machine learning, and segment patients into risk groups using unsupervised learning (K-Means clustering).

## Workflow
1. Data Loading & Target Variable Creation
2. Feature Engineering
3. Model Training — Logistic Regression
4. Model Training — Random Forest
5. Data Leakage Detection & Correction
6. Feature Engineering — Lead Time
7. Cross-Validation
8. Statistical Testing (Chi-Square)
9. Hyperparameter Tuning (GridSearchCV)
10. Patient Risk Segmentation (K-Means + PCA)

## 1. Data Loading & Target Variable Creation

Load the cleaned healthcare master dataset and create the binary target variable:  
- `target = 1` → Patient did **not** show up (No-Show)  
- `target = 0` → Patient attended the appointment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/healthcare_master.csv')

print('Dataset Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head()

In [ ]:
# Create binary target variable
df['target'] = df['status'].apply(lambda x: 1 if x == 'No-show' else 0)

print('Target Distribution:')
print(df['target'].value_counts())
print(f'\nNo-Show Rate: {df["target"].mean()*100:.1f}%')

## 2. Feature Engineering & Train-Test Split

Select relevant features for the model. Categorical features are one-hot encoded using `pd.get_dummies()`.  
Data is split 80/20 for training and testing.

In [ ]:
from sklearn.model_selection import train_test_split

features = ['gender', 'age_group', 'priority', 'reason_for_visit', 'treatment_type']

X = pd.get_dummies(df[features], drop_first=True)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

## 3. Logistic Regression — Baseline Model

Train a baseline Logistic Regression model to establish initial performance benchmarks.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

print('=== Logistic Regression Results ===')
print(f'Accuracy: {accuracy_score(y_test, lr_preds):.3f}')
print('\nClassification Report:')
print(classification_report(y_test, lr_preds))

ConfusionMatrixDisplay.from_predictions(y_test, lr_preds)
plt.title('Confusion Matrix — Logistic Regression')
plt.show()

In [ ]:
# Feature importance via coefficient magnitude
importance_lr = abs(lr_model.coef_[0])
pd.Series(importance_lr, index=X.columns).sort_values().plot(kind='barh', figsize=(8, 5))
plt.title('Feature Importance — Logistic Regression')
plt.tight_layout()
plt.show()

## 4. Random Forest Classifier

Train a Random Forest model — an ensemble method that typically outperforms Logistic Regression on tabular data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print('=== Random Forest Results (All Features) ===')
print(f'Accuracy:  {accuracy_score(y_test, rf_preds):.3f}')
print(f'Precision: {precision_score(y_test, rf_preds):.3f}')
print(f'Recall:    {recall_score(y_test, rf_preds):.3f}')
print(f'F1 Score:  {f1_score(y_test, rf_preds):.3f}')

cm_rf = confusion_matrix(y_test, rf_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='viridis')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Random Forest')
plt.savefig('rf_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance plot
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Feature Importance — Random Forest')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 Features:')
print(importances.head())

## 5. ⚠️ Data Leakage Detection & Correction

**Critical Finding:** Feature importance analysis revealed that `priority` (Low/Medium) was the dominant predictor.

Investigation showed that `priority = Low` and `priority = Medium` **perfectly predicted no-show = 0** in 100% of cases — this is **data leakage**, not a genuine behavioral signal.

**Action:** Remove `priority` from features and retrain for an honest evaluation.

In [ ]:
# Confirm leakage
print('Priority vs Target distribution:')
print(df.groupby('priority')['target'].value_counts(normalize=True))
print('\n=> priority = Low/Medium always maps to target = 0 (No leakage = 100% signal)')

In [ ]:
# Remove priority columns from features
X_train_clean = X_train.drop(columns=[col for col in X_train.columns if col.startswith('priority_')])
X_test_clean  = X_test.drop(columns=[col for col in X_test.columns  if col.startswith('priority_')])

rf_clean = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clean.fit(X_train_clean, y_train)
preds_clean = rf_clean.predict(X_test_clean)

print('=== Random Forest — Leakage Corrected ===')
print(f'Accuracy:  {accuracy_score(y_test, preds_clean):.3f}')
print(f'Precision: {precision_score(y_test, preds_clean):.3f}')
print(f'Recall:    {recall_score(y_test, preds_clean):.3f}')
print(f'F1 Score:  {f1_score(y_test, preds_clean):.3f}')
print('\nNote: Accuracy drop from ~77% to ~50% confirms priority was leaking the target.')

## 6. Feature Engineering — Lead Time Days

Engineer a new feature: `lead_time_days` = days between patient registration and appointment date.  
Patients booked far in advance may be more likely to forget or cancel.

In [ ]:
df['lead_time_days'] = (
    pd.to_datetime(df['appointment_date']) - pd.to_datetime(df['registration_date'])
).dt.days

print('Lead Time Days — Summary Statistics:')
print(df['lead_time_days'].describe())

X_train_v2 = X_train_clean.copy()
X_test_v2  = X_test_clean.copy()
X_train_v2['lead_time_days'] = df.loc[X_train_v2.index, 'lead_time_days']
X_test_v2['lead_time_days']  = df.loc[X_test_v2.index,  'lead_time_days']

rf_v2 = RandomForestClassifier(n_estimators=100, random_state=42)
rf_v2.fit(X_train_v2, y_train)
preds_v2 = rf_v2.predict(X_test_v2)

print('\n=== Random Forest — With Lead Time Feature ===')
print(f'Accuracy:  {accuracy_score(y_test, preds_v2):.3f}')
print(f'Precision: {precision_score(y_test, preds_v2):.3f}')
print(f'Recall:    {recall_score(y_test, preds_v2):.3f}')
print(f'F1 Score:  {f1_score(y_test, preds_v2):.3f}')

## 7. Cross-Validation (5-Fold)

Use 5-fold cross-validation for a more reliable performance estimate — avoids overfitting to a single train-test split.

In [ ]:
from sklearn.model_selection import cross_val_score

X_v2 = pd.concat([X_train_v2, X_test_v2])
y_v2 = pd.concat([y_train, y_test])

scores = cross_val_score(rf_v2, X_v2, y_v2, cv=5, scoring='accuracy')

print('5-Fold Cross-Validation Results:')
print(f'  Fold Scores: {scores.round(3)}')
print(f'  Mean Accuracy: {scores.mean():.3f}')
print(f'  Std Dev:       {scores.std():.3f}')
print('\nInterpretation: ~62.5% accuracy reflects honest model performance on a 200-record dataset.')

## 8. Statistical Validation — Chi-Square Test

Test whether `reason_for_visit` is statistically associated with no-show outcomes using a chi-square test of independence.

In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['reason_for_visit'], df['target'])
chi2, p, dof, expected = chi2_contingency(contingency)

print('Chi-Square Test: reason_for_visit vs No-Show')
print(f'  Chi2 Statistic: {chi2:.3f}')
print(f'  p-value:        {p:.3f}')
print(f'  Degrees of Freedom: {dof}')
print(f'\nConclusion: p = {p:.3f} > 0.05 => NOT statistically significant.')
print('reason_for_visit alone does not strongly predict no-show behavior.')

## 9. Hyperparameter Tuning — GridSearchCV

Use GridSearchCV to find the optimal Random Forest configuration. Tuning parameters: `n_estimators` and `max_depth`.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth':    [3, 5, 7, None]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1'
)
grid.fit(X_v2, y_v2)

print('GridSearchCV Results:')
print(f'  Best Parameters: {grid.best_params_}')
print(f'  Best F1 Score:   {grid.best_score_:.3f}')
print('\nConclusion: Low F1 confirms the bottleneck is dataset size & feature richness, not model config.')

## 10. Patient Risk Segmentation — K-Means Clustering

Use K-Means (k=3) to segment patients into **Low Risk**, **Medium Risk**, and **High Risk** groups.  
PCA is used to reduce dimensions to 2D for visualization.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Segment using age only (small dataset)
seg_data = df[['age']]
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Patient_Segment'] = kmeans.fit_predict(seg_data)

segment_map = {0: 'Low Risk', 1: 'Medium Risk', 2: 'High Risk'}
df['Patient_Segment'] = df['Patient_Segment'].map(segment_map)

print('Patient Segment Distribution:')
print(df['Patient_Segment'].value_counts())

In [ ]:
# PCA visualization of clusters
viz_features = df[['age', 'cost', 'amount']]
scaler = StandardScaler()
viz_scaled = scaler.fit_transform(viz_features)

pca = PCA(n_components=2)
reduced = pca.fit_transform(viz_scaled)

segment_codes = df['Patient_Segment'].astype('category').cat.codes
categories    = df['Patient_Segment'].astype('category').cat.categories.tolist()

plt.figure(figsize=(7, 5))
scatter = plt.scatter(reduced[:, 0], reduced[:, 1], c=segment_codes, cmap='viridis', alpha=0.7)
plt.colorbar(scatter, ticks=[0, 1, 2], label='Segment')
plt.title('Patient Risk Segmentation — K-Means (PCA Visualization)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(handles=scatter.legend_elements()[0], labels=categories)
plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save segmented dataset
df.to_csv('healthcare_segmented.csv', index=False)
print('Segmented dataset saved to healthcare_segmented.csv')

# Save predictions
df_predictions = X_test_v2.copy()
df_predictions['Actual']    = y_test.values
df_predictions['Predicted'] = preds_v2
df_predictions.to_csv('predicted_no_show.csv', index=False)
print('Predictions saved to predicted_no_show.csv')

---
## Summary of Results

| Model | Notes | Accuracy | F1 Score |
|---|---|---|---|
| Logistic Regression | All features (with priority) | 0.675 | 0.380 |
| Random Forest | All features (with priority) | 0.775 | 0.526 |
| Random Forest | Leakage-corrected | 0.500 | 0.231 |
| Random Forest | + lead_time_days | 0.600 | 0.111 |
| Random Forest (5-fold CV) | Final honest estimate | 0.625 ± 0.047 | — |

### Key Takeaways
- **Data leakage** in `priority` feature was identified and corrected — critical ML skill
- Honest model accuracy ~62.5% reflects dataset size limitation (200 records)
- `reason_for_visit` is **not statistically significant** (chi-square p = 0.182)
- GridSearchCV confirms bottleneck is **data**, not model configuration
- K-Means identified **32 High-Risk patients** for proactive intervention